# 01 — Build Splits

Genera los datasets `train.csv`, `val.csv` y `test.csv` a partir de los tres pools de datos
según el esquema definido en `configs/dataset.yaml`.

| Split | Raw | Positivos | Hard Neg | Total | % DC |
|-------|-----|-----------|----------|-------|------|
| Train | 49k | 13k | 4.9k | **66.9k** | 20.53% |
| Val   | 21k | 1k  | 2.1k | **24.1k** | 5.46%  |
| Test  | 30k | —   | —    | **30k**   | 1.50%  |

> ⚠️ `test.csv` se genera aquí pero **NO debe usarse hasta Sprint 10**.

## 0 — Setup

In [ ]:
# Correr solo en Colab
import os
IN_COLAB = 'google.colab' in str(get_ipython().extension_manager.loaded) if hasattr(get_ipython(), 'extension_manager') else False

if IN_COLAB:
    exec(open('/content/drive/MyDrive/contact-detection/scripts/colab_setup.py').read())
    REPO_DIR   = '/content/meli-contact-detection'
    POOLS_DIR  = '/content/drive/MyDrive/contact-detection/data/pools'
    SPLITS_DIR = '/content/drive/MyDrive/contact-detection/data/splits'
    CONFIG     = f'{REPO_DIR}/configs/dataset.yaml'
else:
    # Local
    import sys
    REPO_DIR   = '..'
    sys.path.insert(0, REPO_DIR)
    POOLS_DIR  = '../data/pools'
    SPLITS_DIR = '../data/splits'
    CONFIG     = '../configs/dataset.yaml'

print(f'POOLS_DIR  = {POOLS_DIR}')
print(f'SPLITS_DIR = {SPLITS_DIR}')
print(f'CONFIG     = {CONFIG}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import yaml
from pathlib import Path

from src.data.build_splits import build_splits, load_config

cfg = load_config(CONFIG)
print('Config cargado — seed:', cfg['seed'])

## 1 — Inspección de los pools (antes de mezclar)

In [ ]:
pools = {
    'raw':       pd.read_csv(Path(POOLS_DIR) / cfg['pools']['raw']['file']),
    'positivos': pd.read_csv(Path(POOLS_DIR) / cfg['pools']['positive']['file']),
    'hard_neg':  pd.read_csv(Path(POOLS_DIR) / cfg['pools']['hard_negative']['file']),
}

label_col = 'RESULT'

summary = pd.DataFrame([
    {
        'Pool': name,
        'Total': f"{len(df):,}",
        'DC=1': int(df[label_col].sum()),
        'DC=0': int((df[label_col] == 0).sum()),
        '% DC': f"{df[label_col].mean():.2%}",
    }
    for name, df in pools.items()
])
print(summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
colors = ['#2196F3', '#F44336']

for ax, (name, df) in zip(axes, pools.items()):
    counts = df[label_col].value_counts().sort_index()
    ax.bar(['DC=0', 'DC=1'], counts.values, color=colors)
    ax.set_title(f'Pool: {name}\n{len(df):,} casos  |  {df[label_col].mean():.2%} DC')
    ax.set_ylabel('Cantidad')
    for i, v in enumerate(counts.values):
        ax.text(i, v + len(df)*0.005, f'{v:,}', ha='center', fontsize=9)

plt.suptitle('Distribución de clases por pool', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 2 — Ejecutar build_splits

In [ ]:
build_splits(
    config_path=CONFIG,
    pools_dir=POOLS_DIR,
    output_dir=SPLITS_DIR,
)

## 3 — Verificación de los splits generados

In [ ]:
splits_dir = Path(SPLITS_DIR)

train = pd.read_csv(splits_dir / 'train.csv')
val   = pd.read_csv(splits_dir / 'val.csv')
test  = pd.read_csv(splits_dir / 'test.csv')

print('Filas:')
print(f'  train : {len(train):>7,}  (esperado: 66,900)')
print(f'  val   : {len(val):>7,}  (esperado: 24,100)')
print(f'  test  : {len(test):>7,}  (esperado: 30,000)')
print()
print('Tasa DC:')
print(f'  train : {train[label_col].mean():.4f}  (esperado: 0.2053)')
print(f'  val   : {val[label_col].mean():.4f}  (esperado: 0.0546)')
print(f'  test  : {test[label_col].mean():.4f}  (esperado: 0.0150)')

In [ ]:
# Comparación visual: distribución esperada vs real
expected = {
    'train': cfg['expected']['train']['dc_rate'],
    'val':   cfg['expected']['val']['dc_rate'],
    'test':  cfg['expected']['test']['dc_rate'],
}
actual = {
    'train': train[label_col].mean(),
    'val':   val[label_col].mean(),
    'test':  test[label_col].mean(),
}

x = np.arange(3)
width = 0.35
labels = ['train', 'val', 'test']

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, [expected[l] for l in labels], width, label='Esperado', color='#90CAF9')
bars2 = ax.bar(x + width/2, [actual[l]   for l in labels], width, label='Real',     color='#1565C0')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_ylabel('Tasa DC')
ax.set_title('Tasa de DC esperada vs real por split')
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.2%}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.2%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 4 — Verificación de no solapamiento

In [ ]:
id_col = 'item_id'

ids_train = set(train[id_col])
ids_val   = set(val[id_col])
ids_test  = set(test[id_col])

overlaps = {
    'train ∩ val':  len(ids_train & ids_val),
    'train ∩ test': len(ids_train & ids_test),
    'val ∩ test':   len(ids_val   & ids_test),
}

for pair, n in overlaps.items():
    status = '✅ OK' if n == 0 else f'❌ SOLAPAMIENTO: {n} items'
    print(f'  {pair:<18} {status}')

## 5 — Resumen final

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
titles = [
    f'train  ({len(train):,} filas)',
    f'val    ({len(val):,} filas)',
    f'test   ({len(test):,} filas)\n[NO usar hasta Sprint 10]',
]
colors_split = ['#1B5E20', '#F57F17', '#B71C1C']

for ax, df, title, color in zip(axes, [train, val, test], titles, colors_split):
    counts = df[label_col].value_counts().sort_index()
    bars = ax.bar(['DC=0', 'DC=1'], counts.values, color=[color + '55', color], edgecolor=color)
    ax.set_title(title, fontsize=10)
    dc_rate = df[label_col].mean()
    ax.set_xlabel(f'Tasa DC = {dc_rate:.2%}')
    for bar, val_n in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + len(df)*0.003,
                f'{val_n:,}', ha='center', fontsize=9)

plt.suptitle('Distribución final de los splits', fontsize=13)
plt.tight_layout()
plt.show()

total = len(train) + len(val) + len(test)
print(f'Total filas en dataset: {total:,}')
print(f'DC rate global:         {(train[label_col].sum() + val[label_col].sum() + test[label_col].sum()) / total:.2%}')

---
**Siguiente paso:** `02_baseline_10imgs.ipynb` — Fine-tuning multimodal con el split de train.